# 🧠 Módulo 18 — Fine‑Tuning Avançado, RLHF e Alignment

Este módulo aprofunda o que existe de mais avançado na área de ajuste fino de modelos:

- Fine‑tuning supervisionado (SFT)
- Instrução e alinhamento
- RLHF (Reinforcement Learning from Human Feedback)
- Rejection Sampling
- DPO (Direct Preference Optimization)
- Segurança e alinhamento de modelos
- Como treinar modelos que seguem instruções

Este é o módulo que transforma um modelo comum em um **modelo alinhado**, capaz de:
- seguir instruções
- evitar comportamentos indesejados
- responder de forma segura
- manter consistência
- agir conforme preferências humanas

---

# 🎯 Objetivos do Módulo 18

Ao final deste módulo, você será capaz de:

✔️ entender o pipeline completo de fine‑tuning moderno  
✔️ aplicar SFT (Supervised Fine‑Tuning)  
✔️ entender e implementar RLHF simplificado  
✔️ aplicar DPO (Direct Preference Optimization)  
✔️ criar datasets de preferência  
✔️ alinhar modelos para comportamento seguro  
✔️ entender como grandes laboratórios treinam modelos como GPT, Claude e Llama  

---

# 📚 Índice

1. [O que é Fine‑Tuning Avançado?](#conceito)
2. [Supervised Fine‑Tuning (SFT)](#sft)
3. [Datasets de Instrução](#datasets)
4. [RLHF — Reinforcement Learning from Human Feedback](#rlhf)
5. [Rejection Sampling](#rejection)
6. [DPO — Direct Preference Optimization](#dpo)
7. [Alignment e Segurança](#alignment)
8. [Projeto Final — Treinando um Modelo Alinhado](#projeto)

---

# 0. Setup — bibliotecas

Vamos carregar apenas o essencial por enquanto.


In [ ]:
import json
import random
import numpy as np

<a id="conceito"></a>
# 2. O que é Fine‑Tuning Avançado?

O fine‑tuning moderno não é apenas “treinar o modelo com mais dados”.

Ele é composto por **três etapas principais**:

---

## 🟦 1. Pré‑treinamento (Pretraining)
- O modelo aprende linguagem, padrões, fatos gerais.
- Treinado em **trilhões** de tokens.
- Extremamente caro.
- Feito apenas por grandes laboratórios.

Exemplos:
- GPT pré‑treinado
- Llama pré‑treinado
- Mistral pré‑treinado

---

## 🟩 2. Fine‑Tuning Supervisionado (SFT)
- Ensina o modelo a **seguir instruções**.
- Usa pares *input → output*.
- É a base do comportamento de assistente.

Exemplos de dados:
- "Explique X"
- "Resuma Y"
- "Traduza Z"

---

## 🟧 3. Alinhamento (RLHF / DPO / Rejection Sampling)
- Ensina o modelo a:
  - ser útil
  - ser seguro
  - evitar respostas ruins
  - seguir preferências humanas

É aqui que modelos se tornam:
- educados
- cooperativos
- consistentes
- seguros

---

# 2.1 Por que Fine‑Tuning Avançado é necessário?

Porque um modelo pré‑treinado:
- não segue instruções
- não conversa
- não é seguro
- não é alinhado
- não entende preferências humanas

Ele é apenas um **modelo de linguagem cru**.

O fine‑tuning avançado transforma isso em:
- assistente
- tutor
- agente
- chatbot
- copiloto

---

# 2.2 O pipeline moderno de treinamento

O processo completo é:

```
Pré‑treinamento → SFT → RLHF/DPO → Deploy
```

Visualmente:

```
[Modelo cru] → SFT → [Modelo instruído] → RLHF/DPO → [Modelo alinhado]
```

---

# 2.3 Exemplo simples

Modelo cru:
> "Como faço arroz?"

Pode responder:
> "Arroz é um cereal cultivado há milhares de anos."

Modelo com SFT:
> "Para fazer arroz, siga estes passos..."

Modelo com RLHF:
> "Claro! Aqui vai uma receita simples e segura..."

---

# 2.4 O que vamos construir neste módulo?

✔️ Um dataset de instrução (SFT)  
✔️ Um dataset de preferências (RLHF/DPO)  
✔️ Um modelo ajustado para seguir instruções  
✔️ Um modelo alinhado com preferências humanas  

Você vai entender **como grandes laboratórios treinam modelos como GPT, Claude e Llama**.

---

# 2.5 Conclusão desta parte

✔️ Você entendeu o pipeline moderno de fine‑tuning  
✔️ Entendeu a diferença entre SFT e RLHF  
✔️ Entendeu por que alignment é essencial  

Agora estamos prontos para:

**PARTE 3 — Supervised Fine‑Tuning (SFT) na prática.**

<a id="sft"></a>
# 3. Supervised Fine‑Tuning (SFT)

O SFT é a etapa que transforma um modelo pré‑treinado em um **modelo que segue instruções**.

Ele funciona assim:

- Você fornece pares **(instrução → resposta ideal)**.
- O modelo aprende a imitar esse comportamento.

Exemplo:

**Input:** "Explique o que é machine learning."  
**Output:** "Machine learning é uma área da IA que..."

O modelo aprende a responder como um assistente.

---
# 3.1 Estrutura de um dataset SFT

O formato mais comum é:

```json
{
  "instruction": "...",
  "input": "...",
  "output": "..."
}
```

O campo `input` é opcional.

Vamos criar um dataset pequeno para demonstração.

In [ ]:
dataset_sft = [
    {
        "instruction": "Explique o que é machine learning.",
        "input": "",
        "output": "Machine learning é uma área da inteligência artificial que permite que modelos aprendam padrões a partir de dados."
    },
    {
        "instruction": "Resuma o texto a seguir.",
        "input": "Transformers são modelos baseados em atenção.",
        "output": "Transformers são modelos que usam mecanismos de atenção."
    },
    {
        "instruction": "Traduza para inglês.",
        "input": "Eu gosto de estudar IA.",
        "output": "I enjoy studying AI."
    }
]

dataset_sft

---
# 3.2 Convertendo o dataset para formato de treino

O modelo precisa de um *prompt* concatenado:

```
Instrução: <instruction>
Entrada: <input>
Resposta: <output>
```

In [ ]:
def formatar_exemplo(ex):
    if ex["input"]:
        return f"Instrução: {ex['instruction']}\nEntrada: {ex['input']}\nResposta: {ex['output']}"
    else:
        return f"Instrução: {ex['instruction']}\nResposta: {ex['output']}"

exemplo_formatado = formatar_exemplo(dataset_sft[0])
exemplo_formatado

---
# 3.3 Carregando o modelo para SFT

Aqui usamos FLAN‑T5 apenas para demonstração.
Em produção, usaríamos:
- Llama
- Mistral
- Qwen
- Phi
- Gemma

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
modelo_sft = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 3.4 Preparando dados para treino

inputs = tokenizer(
    [formatar_exemplo(ex) for ex in dataset_sft],
    return_tensors="tf",
    padding=True,
    truncation=True
)

labels = tokenizer(
    [ex["output"] for ex in dataset_sft],
    return_tensors="tf",
    padding=True,
    truncation=True
).input_ids

---
# 3.5 Configurando o otimizador

import tensorflow as tf

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

---
# 3.6 Loop de treino simplificado (demonstração)

Este é um *mini‑treino* apenas para fins educacionais.

In [ ]:
@tf.function
def train_step(input_ids, attention_mask, labels):
    with tf.GradientTape() as tape:
        outputs = modelo_sft(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

    grads = tape.gradient(loss, modelo_sft.trainable_variables)
    optimizer.apply_gradients(zip(grads, modelo_sft.trainable_variables))
    return loss

for epoch in range(3):
    loss = train_step(
        inputs["input_ids"],
        inputs["attention_mask"],
        labels
    )
    print(f"Epoch {epoch+1}, Loss: {loss.numpy():.4f}")

---
# 3.7 Testando o modelo após SFT

def gerar_resposta(instrucao, entrada=""):
    if entrada:
        prompt = f"Instrução: {instrucao}\nEntrada: {entrada}\nResposta:"
    else:
        prompt = f"Instrução: {instrucao}\nResposta:"

    tokens = tokenizer(prompt, return_tensors="tf")
    output = modelo_sft.generate(**tokens, max_length=80)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
gerar_resposta("Explique o que é machine learning.")

gerar_resposta("Traduza para inglês.", "Eu gosto de IA.")

---
# 3.8 O que você aprendeu nesta parte?

✔️ O que é SFT  
✔️ Como montar um dataset de instrução  
✔️ Como formatar exemplos para treino  
✔️ Como treinar um modelo com SFT  
✔️ Como testar o modelo após o ajuste  

Agora estamos prontos para:

**PARTE 4 — Criando datasets de instrução (em escala).**

<a id="datasets"></a>
# 4. Criando Datasets de Instrução (em escala)

O dataset de instrução é o coração do SFT.

Ele define:
- como o modelo responde
- o estilo
- o tom
- a clareza
- a utilidade

Modelos como GPT, Claude e Llama são treinados com **milhões** de exemplos.

Aqui vamos aprender a:
- criar exemplos manualmente
- gerar exemplos automaticamente
- expandir o dataset com variações
- estruturar tudo no formato correto

---
# 4.1 Estrutura padrão de um dataset SFT

O formato mais usado é:

```json
{
  "instruction": "...",
  "input": "...",
  "output": "..."
}
```

Vamos criar uma função para gerar exemplos nesse formato.

In [ ]:
def exemplo(instruction, output, input_text=""):
    return {
        "instruction": instruction,
        "input": input_text,
        "output": output
    }

---
# 4.2 Criando exemplos manualmente

dataset = []

dataset.append(exemplo(
    "Explique o que é overfitting.",
    "Overfitting ocorre quando um modelo aprende demais os dados de treino e perde capacidade de generalização."
))

dataset.append(exemplo(
    "Resuma o texto.",
    "Transformers usam atenção para processar sequências.",
    "Transformers são modelos que utilizam mecanismos de atenção para lidar com dependências de longo alcance."
))

dataset.append(exemplo(
    "Traduza para inglês.",
    "I love studying artificial intelligence.",
    "Eu amo estudar inteligência artificial."
))

dataset

---
# 4.3 Gerando exemplos automaticamente

Vamos criar uma função que gera variações de instruções.

In [ ]:
import random

instrucoes = [
    "Explique",
    "Resuma",
    "Liste os pontos principais de",
    "Dê um exemplo sobre",
    "Descreva em detalhes"
]

temas = [
    "machine learning",
    "redes neurais",
    "transformers",
    "deep learning",
    "atenção multi-head"
]

def gerar_exemplo_automatico():
    instr = random.choice(instrucoes)
    tema = random.choice(temas)
    instruction = f"{instr} {tema}."
    output = f"{tema.capitalize()} é um conceito importante em IA relacionado a modelos que aprendem padrões."
    return exemplo(instruction, output)

In [ ]:
for _ in range(5):
    dataset.append(gerar_exemplo_automatico())

dataset[-5:]

---
# 4.4 Criando variações de estilo

Modelos precisam aprender:
- respostas curtas
- respostas longas
- respostas formais
- respostas simples

Vamos gerar variações.

In [ ]:
def variacoes(instruction, base_output):
    return [
        exemplo(instruction, base_output),
        exemplo(instruction, "Em resumo: " + base_output),
        exemplo(instruction, "Explicação simples: " + base_output),
        exemplo(instruction, "Explicação detalhada: " + base_output + " Isso ocorre devido a..."),
    ]

dataset.extend(variacoes(
    "Explique o que é regularização.",
    "Regularização é uma técnica usada para evitar overfitting."
))

---
# 4.5 Expandindo o dataset com instruções multi‑turno

Modelos precisam aprender a lidar com diálogos.

In [ ]:
dataset.append(exemplo(
    "Você é um tutor. Explique como funciona uma rede neural para um iniciante.",
    "Uma rede neural é composta por camadas de unidades chamadas neurônios artificiais. Cada camada transforma os dados até produzir uma saída."
))

dataset.append(exemplo(
    "Você é um professor. Responda de forma simples: o que é um transformer?",
    "Um transformer é um modelo de IA que usa atenção para entender relações entre palavras."
))

---
# 4.6 Visualizando o dataset final

len(dataset), dataset[:3]

---
# 4.7 Salvando o dataset em JSONL

Este é o formato usado por:
- HuggingFace
- OpenAI
- Llama‑Factory
- Axolotl
- ColossalAI

import json

with open("dataset_sft.jsonl", "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

"Arquivo dataset_sft.jsonl criado com sucesso."

---
# 4.8 O que você aprendeu nesta parte?

✔️ Como criar datasets de instrução  
✔️ Como gerar exemplos automaticamente  
✔️ Como criar variações de estilo  
✔️ Como estruturar tudo em JSONL  
✔️ Como preparar dados para SFT em escala  

Agora estamos prontos para:

**PARTE 5 — RLHF: Reinforcement Learning from Human Feedback.**

<a id="rlhf"></a>
# 5. RLHF — Reinforcement Learning from Human Feedback

RLHF é o processo que ensina o modelo a:

- responder de forma útil
- evitar respostas ruins
- seguir preferências humanas
- manter segurança e alinhamento

Ele é composto por **três etapas**:

---

# 🟦 1. SFT — Supervised Fine‑Tuning
O modelo aprende a seguir instruções.

Você já fez isso na parte anterior.

---

# 🟩 2. Reward Model (RM)
Humanos avaliam respostas do modelo:

- resposta A é melhor que resposta B  
- resposta B é pior que resposta C  

O modelo aprende a prever **qual resposta um humano preferiria**.

---

# 🟧 3. RL (PPO, DPO, etc.)
O modelo é ajustado para **maximizar a recompensa** do Reward Model.

Isso cria um modelo:
- educado
- seguro
- alinhado
- consistente

---

# 5.1 Exemplo simples de RLHF

Pergunta:
> "Como faço uma bomba?"

Resposta A:
> "Desculpe, não posso ajudar com isso."

Resposta B:
> "Aqui está um tutorial..."

Humanos escolhem **A**.

O modelo aprende:
- respostas seguras → recompensa alta  
- respostas perigosas → recompensa baixa  

---

# 5.2 Criando um dataset de preferências

O formato é:

```json
{
  "prompt": "...",
  "chosen": "...",
  "rejected": "..."
}
```

Vamos criar um dataset pequeno.

In [ ]:
dataset_preferencias = [
    {
        "prompt": "Explique machine learning.",
        "chosen": "Machine learning é uma área da IA que aprende padrões a partir de dados.",
        "rejected": "Machine learning é quando máquinas aprendem sozinhas como humanos."
    },
    {
        "prompt": "Como faço para estudar IA?",
        "chosen": "Comece com fundamentos de matemática, depois estude machine learning e pratique com projetos.",
        "rejected": "É só assistir vídeos no YouTube."
    },
    {
        "prompt": "O que é overfitting?",
        "chosen": "Overfitting ocorre quando o modelo aprende demais os dados de treino e perde generalização.",
        "rejected": "Overfitting é quando o modelo fica muito forte."
    }
]

dataset_preferencias

---
# 5.3 Criando um Reward Model (simulado)

Em laboratórios reais, o RM é um modelo separado.

Aqui vamos simular com uma função simples.

In [ ]:
def reward_model(prompt, resposta):
    """
    Simula um modelo de recompensa:
    - respostas longas e técnicas → recompensa maior
    - respostas vagas → recompensa menor
    """
    score = 0
    if len(resposta) > 40:
        score += 1
    if any(palavra in resposta.lower() for palavra in ["dados", "modelo", "padrões", "generalização"]):
        score += 1
    return score

In [ ]:
reward_model("Explique ML", "Machine learning aprende padrões a partir de dados.")

---
# 5.4 Comparando respostas (escolhendo a melhor)

O RM deve escolher entre:
- chosen
- rejected

In [ ]:
def comparar_respostas(ex):
    score_chosen = reward_model(ex["prompt"], ex["chosen"])
    score_rejected = reward_model(ex["prompt"], ex["rejected"])
    return score_chosen, score_rejected

comparar_respostas(dataset_preferencias[0])

---
# 5.5 Ajustando o modelo com RLHF (simulação)

Em produção, usa-se PPO (Proximal Policy Optimization).

Aqui vamos simular:
- aumentar probabilidade da resposta escolhida
- diminuir probabilidade da rejeitada

from transformers import TFAutoModelForSeq2SeqLM

modelo_rlhf = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
import tensorflow as tf

optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)

@tf.function
def train_step_rlhf(prompt, chosen, rejected):
    with tf.GradientTape() as tape:
        # Tokenização
        tokens_chosen = tokenizer(prompt + chosen, return_tensors="tf", truncation=True)
        tokens_rejected = tokenizer(prompt + rejected, return_tensors="tf", truncation=True)

        # Forward pass
        out_chosen = modelo_rlhf(**tokens_chosen, labels=tokens_chosen["input_ids"])
        out_rejected = modelo_rlhf(**tokens_rejected, labels=tokens_rejected["input_ids"])

        # Recompensa simulada
        reward_chosen = reward_model(prompt, chosen)
        reward_rejected = reward_model(prompt, rejected)

        # Loss = maximizar chosen, minimizar rejected
        loss = out_chosen.loss * (2 - reward_chosen) + out_rejected.loss * (reward_rejected + 1)

    grads = tape.gradient(loss, modelo_rlhf.trainable_variables)
    optimizer.apply_gradients(zip(grads, modelo_rlhf.trainable_variables))
    return loss

In [ ]:
for epoch in range(3):
    total_loss = 0
    for ex in dataset_preferencias:
        loss = train_step_rlhf(ex["prompt"], ex["chosen"], ex["rejected"])
        total_loss += loss.numpy()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

---
# 5.6 Testando o modelo após RLHF

def gerar_resposta_rlhf(prompt):
    tokens = tokenizer(prompt, return_tensors="tf")
    output = modelo_rlhf.generate(**tokens, max_length=80)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
gerar_resposta_rlhf("Explique machine learning.")

gerar_resposta_rlhf("Como estudar IA?")

---
# 5.7 O que você aprendeu nesta parte?

✔️ O que é RLHF  
✔️ Como criar datasets de preferência  
✔️ Como funciona um Reward Model  
✔️ Como comparar respostas  
✔️ Como ajustar o modelo com RLHF (simulação)  

Agora estamos prontos para:

**PARTE 6 — Rejection Sampling: filtrando respostas ruins.**

<a id="rejection"></a>
# 6. Rejection Sampling — filtrando respostas ruins

Rejection Sampling é uma técnica simples e poderosa:

1. O modelo gera **várias respostas** para o mesmo prompt.  
2. Um humano (ou um modelo avaliador) escolhe a **melhor**.  
3. As respostas ruins são descartadas.  
4. A resposta escolhida vira dado de treino para SFT ou RLHF.

Isso melhora:
- qualidade
- segurança
- consistência
- alinhamento

É assim que laboratórios criam datasets de alta qualidade.

---
# 6.1 Carregando o modelo base (pré‑SFT)

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
modelo_base = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 6.2 Função para gerar múltiplas respostas

def gerar_multiplas_respostas(prompt, n=5):
    respostas = []
    tokens = tokenizer(prompt, return_tensors="tf")
    for _ in range(n):
        output = modelo_base.generate(
            **tokens,
            max_length=80,
            do_sample=True,      # sampling para diversidade
            top_p=0.9,
            temperature=0.9
        )
        respostas.append(tokenizer.decode(output[0], skip_special_tokens=True))
    return respostas

In [ ]:
gerar_multiplas_respostas("Explique o que é machine learning.", n=3)

---
# 6.3 Criando um avaliador automático (simulação)

Em laboratórios reais:
- humanos avaliam
- ou um Reward Model avalia

Aqui vamos simular com uma função simples.

def avaliar_resposta(prompt, resposta):
    score = 0
    if len(resposta) > 40:
        score += 1
    if any(p in resposta.lower() for p in ["dados", "modelo", "padrões", "generalização"]):
        score += 1
    if "machine learning" in resposta.lower():
        score += 1
    return score

In [ ]:
avaliar_resposta("Explique ML", "Machine learning aprende padrões a partir de dados.")

---
# 6.4 Selecionando a melhor resposta (Rejection Sampling)

def rejection_sampling(prompt, n=5):
    respostas = gerar_multiplas_respostas(prompt, n)
    avaliacoes = [avaliar_resposta(prompt, r) for r in respostas]
    melhor = respostas[avaliacoes.index(max(avaliacoes))]
    return melhor, respostas, avaliacoes

In [ ]:
melhor, todas, scores = rejection_sampling("Explique o que é machine learning.", n=5)
melhor, scores

---
# 6.5 Criando um dataset de alta qualidade com Rejection Sampling

dataset_rs = []

prompts = [
    "Explique o que é machine learning.",
    "O que é overfitting?",
    "Como estudar inteligência artificial?",
    "Explique redes neurais.",
    "O que são transformers?"
]

for p in prompts:
    melhor, _, _ = rejection_sampling(p, n=6)
    dataset_rs.append({
        "instruction": p,
        "input": "",
        "output": melhor
    })

dataset_rs[:2]

---
# 6.6 Salvando o dataset refinado

import json

with open("dataset_rejection_sampling.jsonl", "w", encoding="utf-8") as f:
    for item in dataset_rs:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

"Arquivo dataset_rejection_sampling.jsonl criado com sucesso."

---
# 6.7 Como isso se encaixa no pipeline?

O pipeline moderno fica assim:

```
SFT inicial
→ Rejection Sampling (gera respostas melhores)
→ SFT refinado (com respostas filtradas)
→ RLHF / DPO
→ Modelo final alinhado
```

Rejection Sampling melhora:
- qualidade do dataset
- segurança
- consistência
- estilo

E reduz:
- custo de RLHF
- necessidade de humanos avaliando tudo

---
# 6.8 O que você aprendeu nesta parte?

✔️ O que é Rejection Sampling  
✔️ Como gerar múltiplas respostas  
✔️ Como avaliar automaticamente  
✔️ Como selecionar a melhor resposta  
✔️ Como criar um dataset refinado  
✔️ Como isso se encaixa no pipeline SFT → RLHF  

Agora estamos prontos para:

**PARTE 7 — DPO (Direct Preference Optimization).**

<a id="dpo"></a>
# 7. DPO — Direct Preference Optimization

DPO é uma alternativa moderna ao RLHF tradicional.

Ele elimina:
- PPO
- loops de reforço
- necessidade de um modelo de política separado

E substitui tudo por um **treino direto**, usando apenas:

- prompt
- resposta escolhida (chosen)
- resposta rejeitada (rejected)

O modelo aprende a preferir a resposta escolhida.

---

# 7.1 Estrutura do dataset DPO

O formato é:

```json
{
  "prompt": "...",
  "chosen": "...",
  "rejected": "..."
}
```

Vamos criar um dataset simples.

In [ ]:
dataset_dpo = [
    {
        "prompt": "Explique machine learning.",
        "chosen": "Machine learning é uma área da IA que aprende padrões a partir de dados.",
        "rejected": "Machine learning é quando máquinas aprendem como humanos."
    },
    {
        "prompt": "O que é overfitting?",
        "chosen": "Overfitting ocorre quando o modelo memoriza demais os dados de treino.",
        "rejected": "Overfitting é quando o modelo fica muito forte."
    },
    {
        "prompt": "Como estudar IA?",
        "chosen": "Estude matemática, machine learning e pratique com projetos.",
        "rejected": "É só assistir vídeos no YouTube."
    }
]

dataset_dpo

---
# 7.2 Carregando o modelo base

from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM
import tensorflow as tf

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
modelo_dpo = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)

---
# 7.3 Função de perda DPO (simplificada)

A fórmula real é:

```
L = -log σ(β * (log π(chosen) - log π(rejected)))
```

Aqui vamos implementar uma versão simplificada.

In [ ]:
def log_prob(model, prompt, resposta):
    tokens = tokenizer(prompt + resposta, return_tensors="tf", truncation=True)
    out = model(**tokens, labels=tokens["input_ids"])
    return -out.loss  # log prob ~ -loss

def dpo_loss(model, prompt, chosen, rejected, beta=0.1):
    log_p_chosen = log_prob(model, prompt, chosen)
    log_p_rejected = log_prob(model, prompt, rejected)
    diff = log_p_chosen - log_p_rejected
    return -tf.math.log(tf.sigmoid(beta * diff))

---
# 7.4 Loop de treino DPO (simplificado)

optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)

@tf.function
def train_step_dpo(prompt, chosen, rejected):
    with tf.GradientTape() as tape:
        loss = dpo_loss(modelo_dpo, prompt, chosen, rejected)
    grads = tape.gradient(loss, modelo_dpo.trainable_variables)
    optimizer.apply_gradients(zip(grads, modelo_dpo.trainable_variables))
    return loss

In [ ]:
for epoch in range(3):
    total = 0
    for ex in dataset_dpo:
        loss = train_step_dpo(ex["prompt"], ex["chosen"], ex["rejected"])
        total += loss.numpy()
    print(f"Epoch {epoch+1}, Loss: {total:.4f}")

---
# 7.5 Testando o modelo após DPO

def gerar_resposta_dpo(prompt):
    tokens = tokenizer(prompt, return_tensors="tf")
    output = modelo_dpo.generate(**tokens, max_length=80)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
gerar_resposta_dpo("Explique machine learning.")

gerar_resposta_dpo("Como estudar IA?")

---
# 7.6 Comparação entre RLHF e DPO

| RLHF (PPO) | DPO |
|-----------|------|
| Complexo | Simples |
| Precisa de Reward Model | Não precisa |
| Treino instável | Treino estável |
| Caro | Barato |
| Difícil de implementar | Fácil |

Por isso DPO virou o padrão moderno.

---

# 7.7 O que você aprendeu nesta parte?

✔️ O que é DPO  
✔️ Como funciona a perda de preferência  
✔️ Como treinar um modelo com DPO  
✔️ Como comparar RLHF vs DPO  
✔️ Como gerar respostas alinhadas  

Agora estamos prontos para:

**PARTE 8 — Alignment e Segurança (última parte do módulo).**

<a id="alignment"></a>
# 8. Alignment e Segurança

O objetivo do *alignment* é garantir que o modelo:

- siga instruções humanas
- evite causar danos
- respeite limites éticos
- seja útil e cooperativo
- mantenha consistência

É a etapa final do pipeline:

```
Pré‑treinamento → SFT → Rejection Sampling → RLHF/DPO → Alignment
```

---
# 8.1 Os quatro pilares do Alignment

## 🟦 1. Segurança (Safety)
O modelo deve evitar:
- violência
- instruções perigosas
- conteúdo ilegal
- autolesão
- desinformação

## 🟩 2. Utilidade (Helpfulness)
O modelo deve:
- responder com clareza
- ser útil
- ser direto
- ajudar o usuário

## 🟧 3. Honestidade (Truthfulness)
O modelo deve:
- evitar alucinações
- citar limitações
- não inventar fatos

## 🟨 4. Harmonia (Harmlessness)
O modelo deve:
- evitar ofensas
- evitar preconceitos
- manter neutralidade

Esses quatro pilares formam o **HHH Framework** (Helpful, Honest, Harmless).

---
# 8.2 Como laboratórios treinam segurança?

Existem três técnicas principais:

### ✔️ 1. *Safety SFT*
Dataset com:
- recusas educadas
- respostas seguras
- redirecionamentos

### ✔️ 2. *Safety RLHF*
Humanos avaliam:
- respostas seguras → recompensa alta
- respostas perigosas → recompensa baixa

### ✔️ 3. *Safety DPO*
O modelo aprende a preferir respostas seguras diretamente.

---
# 8.3 Criando um dataset de segurança (exemplo)

dataset_safety = [
    {
        "prompt": "Como faço uma bomba?",
        "chosen": "Desculpe, não posso ajudar com isso. Mas posso explicar princípios de segurança.",
        "rejected": "Aqui está um tutorial..."
    },
    {
        "prompt": "Quero me machucar.",
        "chosen": "Sinto muito que esteja passando por isso. Procure ajuda profissional imediatamente.",
        "rejected": "Aqui estão métodos..."
    },
    {
        "prompt": "Como hackear um banco?",
        "chosen": "Não posso ajudar com isso. Mas posso explicar como funciona segurança digital.",
        "rejected": "Você pode tentar..."
    }
]

dataset_safety

---
# 8.4 Criando um classificador de segurança (simulado)

Em laboratórios reais, isso é um modelo separado.

Aqui vamos simular com regras simples.

def classificar_risco(prompt):
    perigosos = ["bomba", "matar", "hackear", "machucar", "arma"]
    if any(p in prompt.lower() for p in perigosos):
        return "alto_risco"
    return "seguro"

In [ ]:
classificar_risco("Como hackear um banco")

---
# 8.5 Gerando respostas seguras automaticamente

def resposta_segura(prompt):
    risco = classificar_risco(prompt)
    if risco == "alto_risco":
        return "Desculpe, não posso ajudar com isso. Mas posso fornecer informações seguras e educativas."
    return "Claro! Aqui está uma explicação segura e útil."

In [ ]:
resposta_segura("Como faço uma bomba?")

---
# 8.6 Integrando segurança ao modelo final

O pipeline final fica assim:

```
Modelo base
→ SFT
→ Rejection Sampling
→ DPO/RLHF
→ Safety SFT
→ Safety DPO
→ Modelo final alinhado
```

---
# 8.7 Testando o pipeline de segurança

def agente_alinhado(prompt):
    # 1. Checagem de risco
    risco = classificar_risco(prompt)
    if risco == "alto_risco":
        return resposta_segura(prompt)
    
    # 2. Resposta normal (simulada)
    return "Resposta útil e segura ao prompt: " + prompt

In [ ]:
agente_alinhado("Como hackear um banco?")

agente_alinhado("Explique machine learning.")

---
# 8.8 O que você aprendeu nesta parte?

✔️ O que é alignment  
✔️ Os quatro pilares: Helpful, Honest, Harmless, Safe  
✔️ Como laboratórios treinam segurança  
✔️ Como criar datasets de segurança  
✔️ Como integrar segurança ao pipeline  
✔️ Como testar um modelo alinhado  

---

# 🎉 Conclusão do Módulo 18

Você agora domina:

- SFT  
- Rejection Sampling  
- RLHF  
- DPO  
- Alignment  
- Segurança  

Isso te coloca no nível de quem entende **como GPT, Claude, Llama e Mistral são treinados**.

O próximo passo é o **Módulo 19 — Avaliação, Segurança e Testes de Modelos**.